# FreshGuard v2 - 02 Prepare Detector Data

Inputs required on Kaggle:
- `ulnnproject/food-freshness-dataset`
- notebook 00 output: `freshguard-official-sources-v2`
- notebook 01 output: `freshguard-v2-splits`

Builds a 1-class YOLO `produce` dataset from Food Freshness full-image bootstrap boxes, KTH full-image produce boxes, and Open Images V7 official detection boxes.

In [ ]:
from __future__ import annotations

import json
import shutil
from pathlib import Path

import pandas as pd
from PIL import Image

KAGGLE_INPUT = Path('/kaggle/input')
split_manifests = sorted(KAGGLE_INPUT.rglob('classifier_manifest.csv'))
SPLITS_DIR = split_manifests[0].parent if split_manifests else None
official_summaries = sorted(KAGGLE_INPUT.rglob('official_sources_summary.json'))
OFFICIAL_DIR = official_summaries[0].parent if official_summaries else None
if SPLITS_DIR is None:
    raise FileNotFoundError('Attach notebook 01 output as freshguard-v2-splits.')
if OFFICIAL_DIR is None:
    raise FileNotFoundError('Attach notebook 00 output as freshguard-official-sources-v2.')

OPEN_IMAGES_BOXES = OFFICIAL_DIR / 'open_images_produce' / 'boxes.csv'
if not OPEN_IMAGES_BOXES.exists():
    raise FileNotFoundError(f'Missing official Open Images boxes.csv: {OPEN_IMAGES_BOXES}')

OUT_DIR = Path('/kaggle/working/freshguard_v2_detector_data')
IMAGE_DIR = OUT_DIR / 'images'
LABEL_DIR = OUT_DIR / 'labels'
for split in ('train', 'val', 'test'):
    (IMAGE_DIR / split).mkdir(parents=True, exist_ok=True)
    (LABEL_DIR / split).mkdir(parents=True, exist_ok=True)

classifier_manifest = pd.read_csv(SPLITS_DIR / 'classifier_manifest.csv')
open_images_boxes = pd.read_csv(OPEN_IMAGES_BOXES)
OPEN_IMAGES_IMAGE_ROOT = OFFICIAL_DIR / 'open_images_produce' / 'images'
print({'splits_dir': str(SPLITS_DIR), 'official_dir': str(OFFICIAL_DIR), 'manifest_rows': len(classifier_manifest), 'open_images_boxes': len(open_images_boxes)})

In [ ]:
def yolo_line_from_abs_xyxy(x1: float, y1: float, x2: float, y2: float, width: int, height: int) -> str:
    x1 = max(0.0, min(float(width), x1))
    y1 = max(0.0, min(float(height), y1))
    x2 = max(0.0, min(float(width), x2))
    y2 = max(0.0, min(float(height), y2))
    cx = ((x1 + x2) / 2) / width
    cy = ((y1 + y2) / 2) / height
    bw = (x2 - x1) / width
    bh = (y2 - y1) / height
    return f'0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}'

def copy_image_with_label(src: Path, split: str, label_lines: list[str], prefix: str) -> bool:
    if not src.exists() or split not in {'train', 'val', 'test'}:
        return False
    stem = f'{prefix}_{src.stem}_{abs(hash(str(src))) & 0xffffffff:x}'
    dst_image = IMAGE_DIR / split / f'{stem}{src.suffix.lower()}'
    dst_label = LABEL_DIR / split / f'{stem}.txt'
    shutil.copy2(src, dst_image)
    dst_label.write_text('\n'.join(label_lines) + '\n')
    return True

def resolve_manifest_path(raw_path: str | Path) -> Path | None:
    path = Path(raw_path)
    if path.exists():
        return path
    parts = list(path.parts)
    relative_candidates: list[Path] = []
    for anchor in ('Dataset', 'dataset'):
        if anchor in parts:
            idx = parts.index(anchor)
            relative_candidates.append(Path(*parts[idx:]))
    if len(parts) >= 4:
        relative_candidates.append(Path(*parts[-4:]))
    if len(parts) >= 3:
        relative_candidates.append(Path(*parts[-3:]))
    for rel in relative_candidates:
        for root in KAGGLE_INPUT.iterdir():
            candidate = root / rel
            if candidate.exists():
                return candidate
            nested_matches = list(root.rglob(str(rel).replace('\\', '/')))
            if nested_matches:
                return nested_matches[0]
    matches = list(KAGGLE_INPUT.rglob(path.name))
    return matches[0] if matches else None

def resolve_open_images_path(raw_path: str | Path, split: str) -> Path | None:
    path = Path(raw_path)
    if path.exists():
        return path
    direct = OPEN_IMAGES_IMAGE_ROOT / split / path.name
    if direct.exists():
        return direct
    matches = list((OPEN_IMAGES_IMAGE_ROOT / split).rglob(path.name)) if (OPEN_IMAGES_IMAGE_ROOT / split).exists() else []
    if matches:
        return matches[0]
    all_matches = list(OPEN_IMAGES_IMAGE_ROOT.rglob(path.name)) if OPEN_IMAGES_IMAGE_ROOT.exists() else []
    return all_matches[0] if all_matches else None

def full_image_yolo_line(path: Path) -> str:
    with Image.open(path) as img:
        width, height = img.size
    return yolo_line_from_abs_xyxy(0, 0, width, height, width, height)

In [ ]:
counts = {'food_full_image': 0, 'kth_full_image': 0, 'open_images_boxed': 0}
for row in classifier_manifest.itertuples(index=False):
    path = resolve_manifest_path(row.path)
    if path is None:
        continue
    split = str(row.split)
    source = str(row.source)
    if source == 'food_freshness':
        if copy_image_with_label(path, split, [full_image_yolo_line(path)], 'food'):
            counts['food_full_image'] += 1
    elif source == 'grocery_store_dataset':
        if copy_image_with_label(path, split, [full_image_yolo_line(path)], 'kth'):
            counts['kth_full_image'] += 1

required = {'path', 'split', 'x1', 'y1', 'x2', 'y2', 'coordinate_format'}
missing = required - set(open_images_boxes.columns)
if missing:
    raise ValueError(f'Open Images boxes.csv missing columns: {sorted(missing)}')

for (path_value, split), group in open_images_boxes.groupby(['path', 'split']):
    path = resolve_open_images_path(path_value, str(split))
    if path is None:
        continue
    with Image.open(path) as img:
        width, height = img.size
    label_lines: list[str] = []
    for r in group.itertuples(index=False):
        if str(r.coordinate_format) == 'normalized_xyxy':
            x1, y1, x2, y2 = float(r.x1) * width, float(r.y1) * height, float(r.x2) * width, float(r.y2) * height
        else:
            x1, y1, x2, y2 = float(r.x1), float(r.y1), float(r.x2), float(r.y2)
        if x2 > x1 and y2 > y1:
            label_lines.append(yolo_line_from_abs_xyxy(x1, y1, x2, y2, width, height))
    if label_lines and copy_image_with_label(path, str(split), label_lines, 'oi'):
        counts['open_images_boxed'] += 1

if counts['open_images_boxed'] == 0:
    raise RuntimeError('Open Images contributed zero boxed detector images.')
print(counts)

In [ ]:
data_yaml = OUT_DIR / 'produce.yaml'
data_yaml.write_text('\n'.join([f'path: {OUT_DIR}', 'train: images/train', 'val: images/val', 'test: images/test', 'names:', '  0: produce', '']))
summary = {
    'source_counts': counts,
    'images_by_split': {split: len(list((IMAGE_DIR / split).glob('*'))) for split in ('train', 'val', 'test')},
    'labels_by_split': {split: len(list((LABEL_DIR / split).glob('*.txt'))) for split in ('train', 'val', 'test')},
    'data_yaml': str(data_yaml),
}
(OUT_DIR / 'detector_data_summary.json').write_text(json.dumps(summary, indent=2))
print(summary)
print('Save /kaggle/working/freshguard_v2_detector_data as Kaggle Dataset: freshguard-v2-detector-data')